# Data Preprocessing 1

## Imports and Setup

Load core libraries used for file I/O, JSON parsing, and text processing.

In [1]:
import pandas as pd
import os
import json
import re

## Load Raw Price Data

Read per-ticker CSV files from the StockNet price folder and store them in a dictionary keyed by ticker.

In [2]:

directory = "../data/stocknet-dataset/price/raw"

stock_data = {}

for filename in os.listdir(directory):
    if filename.endswith(".csv"):
        ticker = filename.replace(".csv", "")
        filepath = os.path.join(directory, filename)
        df = pd.read_csv(filepath, parse_dates=["Date"])
        df.set_index("Date", inplace=True)
        stock_data[ticker] = df

print(stock_data["RDS-B"].head())

                 Open       High        Low      Close  Adj Close  Volume
Date                                                                     
2012-09-04  72.379997  72.389999  71.550003  71.830002  53.385010  380900
2012-09-05  71.860001  71.889999  71.449997  71.510002  53.147179  317800
2012-09-06  71.930000  73.339996  71.870003  73.279999  54.462677  541600
2012-09-07  73.269997  73.849998  73.199997  73.750000  54.811977  597600
2012-09-10  73.540001  73.790001  73.110001  73.110001  54.336323  701600


## Create Tidy Price Table

Concatenate all ticker price DataFrames into a single table with a `Ticker` column.

In [3]:
tidy_df = pd.concat(
    [df.assign(Ticker=ticker) for ticker, df in stock_data.items()],
    axis=0
).reset_index()

## Preview Price Data

Quick check of the combined price table.

In [4]:
tidy_df

,Date,Open,High,Low,Close,Adj Close,Volume,Ticker
0,2012-09-04,18.990000,19.139999,18.799999,19.000000,16.202320,27006700.0,CSCO
1,2012-09-05,19.000000,19.120001,18.870001,18.900000,16.117046,30580300.0,CSCO
2,2012-09-06,19.070000,19.750000,19.049999,19.730000,16.824829,59444100.0,CSCO
3,2012-09-07,19.549999,19.650000,19.250000,19.559999,16.679861,44786600.0,CSCO
4,2012-09-10,19.430000,19.469999,19.100000,19.150000,16.330235,40047200.0,CSCO
...,...,...,...,...,...,...,...,...
108587,2017-08-28,39.610001,39.680000,39.299999,39.529999,39.529999,2119900.0,PPL
108588,2017-08-29,39.590000,39.650002,39.360001,39.380001,39.380001,1443900.0,PPL
108589,2017-08-30,39.270000,39.419998,39.110001,39.180000,39.180000,1825100.0,PPL
108590,2017-08-31,39.259998,39.320000,39.169998,39.240002,39.240002,2086200.0,PPL


## Load Raw Tweets

Parse JSON tweet files (supporting both single-JSON and JSON-lines formats) into a unified DataFrame.

In [5]:
root_dir = "../data/stocknet-dataset/tweet/preprocessed"

all_tweets = []

for ticker in os.listdir(root_dir):
    subfolder = os.path.join(root_dir, ticker)
    if os.path.isdir(subfolder):
        for file in os.listdir(subfolder):
            filepath = os.path.join(subfolder, file)
            if os.path.isfile(filepath):
                try:
                    with open(filepath, "r", encoding="utf-8") as f:
                        try:

                            tweet = json.load(f)
                            tweets = [tweet]
                        except json.JSONDecodeError:

                            f.seek(0)
                            tweets = [json.loads(line) for line in f if line.strip()]

                        for tweet in tweets:
                            flat_text = " ".join(tweet.get("text", []))
                            all_tweets.append({
                                "ticker": ticker,
                                "text": flat_text,
                                "created_at": tweet.get("created_at"),
                                "user_id": tweet.get("user_id_str")
                            })
                except Exception as e:
                    print(f"Skipping file {filepath} due to error: {e}")

tweet_df = pd.DataFrame(all_tweets)
tweet_df["created_at"] = pd.to_datetime(tweet_df["created_at"], errors="coerce")

/var/folders/jx/mdk91y8925ncdd32prjkcl_c0000gn/T/ipykernel_68636/1994651854.py:34: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  tweet_df["created_at"] = pd.to_datetime(tweet_df["created_at"], errors="coerce")


## Preview Tweet Data

Inspect the raw tweet DataFrame before further processing.

In [6]:
tweet_df

,ticker,text,created_at,user_id
0,VZ,"$ gdp news : "" actives on open AT_USER $ aapl ...",2014-10-14 13:39:49+00:00,1923902360
1,VZ,rt AT_USER psw / seeking alpha june trade revi...,2015-06-20 14:18:10+00:00,3232889321
2,VZ,rt AT_USER psw / seeking alpha june trade revi...,2015-06-20 16:16:05+00:00,3238685909
3,VZ,rt AT_USER psw / seeking alpha june trade revi...,2015-06-20 16:25:57+00:00,2740607613
4,VZ,rt AT_USER psw / seeking alpha june trade revi...,2015-06-20 16:01:01+00:00,3234872187
...,...,...,...,...
106333,CODI,stock contest ! ! pick $ googl and win a free ...,2014-09-07 00:20:34+00:00,386787305
106334,CODI,codi compass diversified holdings 52wk low cli...,2015-05-18 13:34:15+00:00,2579477766
106335,CODI,$ codi compass diversified holdings files sec ...,2014-08-09 10:38:34+00:00,2717670854
106336,CODI,codi compass diversified holdings financials U...,2015-06-01 19:34:12+00:00,2181403417


## Parse Timestamps

Normalize `created_at` into a timezone-aware datetime and report parse failures.

In [7]:
tweet_df["created_at"] = pd.to_datetime(
    tweet_df["created_at"],
    format="%a %b %d %H:%M:%S %z %Y",
    errors="coerce",
    utc=True
)

# Report parse failures
failed_count = tweet_df["created_at"].isna().sum()
print(f"created_at parse failures: {failed_count}")


created_at parse failures: 0


## Check Parsed Dates

Verify that timestamp parsing worked as expected.

In [8]:
tweet_df

,ticker,text,created_at,user_id
0,VZ,"$ gdp news : "" actives on open AT_USER $ aapl ...",2014-10-14 13:39:49+00:00,1923902360
1,VZ,rt AT_USER psw / seeking alpha june trade revi...,2015-06-20 14:18:10+00:00,3232889321
2,VZ,rt AT_USER psw / seeking alpha june trade revi...,2015-06-20 16:16:05+00:00,3238685909
3,VZ,rt AT_USER psw / seeking alpha june trade revi...,2015-06-20 16:25:57+00:00,2740607613
4,VZ,rt AT_USER psw / seeking alpha june trade revi...,2015-06-20 16:01:01+00:00,3234872187
...,...,...,...,...
106333,CODI,stock contest ! ! pick $ googl and win a free ...,2014-09-07 00:20:34+00:00,386787305
106334,CODI,codi compass diversified holdings 52wk low cli...,2015-05-18 13:34:15+00:00,2579477766
106335,CODI,$ codi compass diversified holdings files sec ...,2014-08-09 10:38:34+00:00,2717670854
106336,CODI,codi compass diversified holdings financials U...,2015-06-01 19:34:12+00:00,2181403417


## Define Text Cleaning

Create a helper to strip URLs, tickers, hashtags, mentions, and punctuation from tweet text.

In [9]:
def clean_text():
    char_patterns = re.compile(
        r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*(),]|(?:%[0-9a-fA-F]{2}))+|'
        r'#[a-zA-Z]+|\$[a-zA-Z]+|@[a-zA-Z]+|[,.^_$*%-;!?:]'
    )
    tweet_df["text"] = tweet_df["text"].str.replace(char_patterns, "", regex=True)


## Extract Date

Derive a `date` column from the parsed datetime for later joins or aggregation.

In [10]:
def date_extract(datetime_obj):
  return str(datetime_obj.date())[:10]

tweet_df['date'] = tweet_df['created_at'].apply(date_extract)

## Review Date Extraction

Inspect tweets after adding the `date` column.

In [11]:
tweet_df

,ticker,text,created_at,user_id,date
0,VZ,"$ gdp news : "" actives on open AT_USER $ aapl ...",2014-10-14 13:39:49+00:00,1923902360,2014-10-14
1,VZ,rt AT_USER psw / seeking alpha june trade revi...,2015-06-20 14:18:10+00:00,3232889321,2015-06-20
2,VZ,rt AT_USER psw / seeking alpha june trade revi...,2015-06-20 16:16:05+00:00,3238685909,2015-06-20
3,VZ,rt AT_USER psw / seeking alpha june trade revi...,2015-06-20 16:25:57+00:00,2740607613,2015-06-20
4,VZ,rt AT_USER psw / seeking alpha june trade revi...,2015-06-20 16:01:01+00:00,3234872187,2015-06-20
...,...,...,...,...,...
106333,CODI,stock contest ! ! pick $ googl and win a free ...,2014-09-07 00:20:34+00:00,386787305,2014-09-07
106334,CODI,codi compass diversified holdings 52wk low cli...,2015-05-18 13:34:15+00:00,2579477766,2015-05-18
106335,CODI,$ codi compass diversified holdings files sec ...,2014-08-09 10:38:34+00:00,2717670854,2014-08-09
106336,CODI,codi compass diversified holdings financials U...,2015-06-01 19:34:12+00:00,2181403417,2015-06-01


## Clean Tweet Text

Apply the text cleaning function and preview results.

In [12]:
clean_text()
tweet_df

,ticker,text,created_at,user_id,date
0,VZ,"gdp news "" actives on open ATUSER aapl tsl...",2014-10-14 13:39:49+00:00,1923902360,2014-10-14
1,VZ,rt ATUSER psw seeking alpha june trade review...,2015-06-20 14:18:10+00:00,3232889321,2015-06-20
2,VZ,rt ATUSER psw seeking alpha june trade review...,2015-06-20 16:16:05+00:00,3238685909,2015-06-20
3,VZ,rt ATUSER psw seeking alpha june trade review...,2015-06-20 16:25:57+00:00,2740607613,2015-06-20
4,VZ,rt ATUSER psw seeking alpha june trade review...,2015-06-20 16:01:01+00:00,3234872187,2015-06-20
...,...,...,...,...,...
106333,CODI,stock contest pick googl and win a free tab...,2014-09-07 00:20:34+00:00,386787305,2014-09-07
106334,CODI,codi compass diversified holdings wk low click...,2015-05-18 13:34:15+00:00,2579477766,2015-05-18
106335,CODI,codi compass diversified holdings files sec f...,2014-08-09 10:38:34+00:00,2717670854,2014-08-09
106336,CODI,codi compass diversified holdings financials U...,2015-06-01 19:34:12+00:00,2181403417,2015-06-01


## Save Processed Outputs

Write the cleaned price and tweet tables to parquet for downstream steps.

In [13]:
os.makedirs("../data/dataset", exist_ok=True)
tidy_df.to_parquet("../data/dataset/stock_prices.parquet", index=False)
tweet_df.to_parquet('../data/dataset/stock_tweets.parquet', index=False)